# 01 – Data Preparation
Load Kepler KOI, keep small-planet candidates around Sun-like stars, create groups.

In [1]:
import pandas as pd
import numpy as np
DATA_PATH = '/kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv'
df = pd.read_csv(DATA_PATH, comment='#')
print('Raw shape:', df.shape)

df = df[df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE', 'CANDIDATE'])].copy()
df = df[df['koi_prad'].between(1.0, 4.0) & df['koi_steff'].between(4700, 6500)].copy()
df = df.dropna(subset=['koi_prad', 'kepid', 'koi_disposition'])
df['group'] = df['kepid']
df['label'] = np.nan
df.loc[df.koi_disposition == 'CONFIRMED', 'label'] = 1
df.loc[df.koi_disposition == 'FALSE POSITIVE', 'label'] = 0
model_df = df[df.label.notna()].copy()
candidate_df = df[df.koi_disposition == 'CANDIDATE'].copy()
model_df['label'] = model_df['label'].astype(int)
print('Filtered:', df.shape, '| labeled:', model_df.shape, '| candidates:', candidate_df.shape)
print(df.koi_disposition.value_counts())
print('Unique host stars:', df.group.nunique())
df.to_csv('/kaggle/working/koi_small_planets_all.csv', index=False)
model_df.to_csv('/kaggle/working/koi_small_planets_labeled.csv', index=False)
candidate_df.to_csv('/kaggle/working/koi_small_planets_candidates.csv', index=False)

Raw shape: (9564, 49)
Filtered: (4249, 51) | labeled: (3195, 51) | candidates: (1054, 51)
koi_disposition
CONFIRMED         1966
FALSE POSITIVE    1229
CANDIDATE         1054
Name: count, dtype: int64
Unique host stars: 3517


In [2]:
assert model_df['group'].notna().all()
assert set(model_df['label'].unique()).issubset({0, 1})
assert model_df['label'].nunique() == 2
assert not (model_df['koi_disposition'] == 'CANDIDATE').any()
print('Data-quality checks passed.')

Data-quality checks passed.
